In [ ]:
import subprocess, sys, torch

# ── 1. Read current torch version ──────────────────────────────────────────
torch_ver = torch.__version__          # e.g. 2.10.0+cu128
cuda_tag  = "cu" + torch_ver.split("cu")[-1] if "cu" in torch_ver else "cu128"
major, minor = torch_ver.split(".")[:2]
tv_minor  = int(minor) - 1             # torch 2.10 → tv 0.25 (minor offset trick)
tv_ver    = f"0.{tv_minor+16}.0"       # 2.10 → 0.25.0  |  2.5 → 0.20.0
index_url = f"https://download.pytorch.org/whl/{cuda_tag}"

print(f"torch  : {torch_ver}")
print(f"target : torchvision=={tv_ver}  index={index_url}")

# ── 2. Remove broken torchvision ───────────────────────────────────────────
subprocess.run([sys.executable, "-m", "pip", "uninstall", "torchvision", "-y"],
               capture_output=True)

# ── 3. Install matching torchvision (no-deps so torch is never touched) ────
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                f"torchvision=={tv_ver}",
                "--no-deps", "--index-url", index_url], check=True)

# ── 4. Install model requirements ──────────────────────────────────────────
pkgs = [
    "transformers==4.45.1",
    "transformers_stream_generator",
    "bitsandbytes>=0.40.0.post4",
    "peft>=0.4.0",
    "sentencepiece",
    "einops",
    "tiktoken",
    "decord",
    "av",
    "imageio",
    "opencv-python",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkgs, check=True)

# ── 5. Verify ───────────────────────────────────────────────────────────────
import importlib
importlib.invalidate_caches()
import torchvision
print(f"\n✓ torch       : {torch.__version__}")
print(f"✓ torchvision : {torchvision.__version__}")

import transformers, peft, bitsandbytes
print(f"✓ transformers: {transformers.__version__}")
print(f"✓ peft        : {peft.__version__}")
print(f"✓ bitsandbytes: {bitsandbytes.__version__}")
print("\n✅ All good — restart the kernel now before running your training script.")

In [ ]:
# !pip install -q av imageio decord opencv-python
# !pip install -q sentence-transformers==2.6.1
# !pip install -q -U bitsandbytes>=0.46.1
# !pip install -q transformers==4.40.1

In [ ]:
import pandas as pd

def generate_csv(filename, max_id):
    df = pd.DataFrame({'video_id': range(max_id + 1)})
    df.to_csv(filename, index=False)

# Usage
generate_csv('/kaggle/working/video_ids.csv', 10)

In [ ]:
"""
=============================================================================
InternVideo2.5  QLoRA Fine-tuning  —  Kaggle T4 ×2
=============================================================================
BUGS FIXED vs original:
  1. DEVICE: force device_map={"": 0} → single GPU, kills ALL cross-device
             errors. 8B in 4-bit ≈ 4 GB + activations fits comfortably on
             one 16 GB T4.
  2. image_flags: was token-level mask [seq_len].
             FIXED → frame-level  [total_frames]  all-ones tensor,
             exactly matching what modeling_internvl_chat_hico2.py expects:
               vit_embeds = vit_embeds[image_flags == 1]   # [n_frames, D]
  3. collate_fn: was torch.stack(image_flags) → wrong shape.
             FIXED → torch.cat along dim=0 → [total_frames_in_batch]
  4. NUM_IMAGE_TOKEN: was hardcoded to 16 (wrong). FIXED → read from
             model.num_image_token after loading, guarantees the number of
             <IMG_CONTEXT> tokens inserted per frame equals what mlp1 outputs.
  5. LoRA device placement: inject_adapter_in_model initialises new LoRA
             parameters on CPU. FIXED → explicit .to(DEVICE) sweep after
             injection, before any forward pass.
  6. gradient checkpointing: manual per-layer wrap via _enable_manual_gradient_checkpointing()
             because InternVLChatModel.supports_gradient_checkpointing=False forbids the API.

=============================================================================
INSTALL (run once before this script):
    pip install -q peft==0.10.0 bitsandbytes>=0.46.1 transformers==4.45.1
    pip install -q einops sentencepiece tiktoken decord av imageio
=============================================================================
"""

# ──────────────────────────────────────────────────────────────────────────
#  Standard library
# ──────────────────────────────────────────────────────────────────────────
import gc
import glob
import json
import math
import os
import random
import warnings

# ──────────────────────────────────────────────────────────────────────────
#  Third-party
# ──────────────────────────────────────────────────────────────────────────
import bitsandbytes as bnb  # FIX D: PagedAdamW8bit
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.transforms as T
from PIL import Image
from peft import LoraConfig, inject_adapter_in_model, prepare_model_for_kbit_training
from torchvision.transforms.functional import InterpolationMode
from tqdm.auto import tqdm
from transformers import AutoConfig, AutoModel, AutoTokenizer, BitsAndBytesConfig

warnings.filterwarnings("ignore")


# ╔═════════════════════════════════════════════════════════════════════════╗
# ║                    ❶  CONFIGURATION  —  edit here                      ║
# ╚═════════════════════════════════════════════════════════════════════════╝

# ── Paths ──────────────────────────────────────────────────────────────────
MODEL_PATH         = "OpenGVLab/InternVideo2_5_Chat_8B"
SAMPLED_FRAMES_DIR = "/kaggle/input/datasets/shresthml/sample-frame/sampled_frames"
ANSWER_CSV_PATH    = "/kaggle/input/datasets/shresthml/ans-csv-forvqa/answer.csv"
REFERENCE_CSV_PATH = "/kaggle/working/video_ids.csv"
OUTPUT_DIR         = "/kaggle/working/qlora_checkpoints"

# ── Device ─────────────────────────────────────────────────────────────────
# FIX 1: Single GPU.  8B × 4-bit ≈ 4 GB.  With activations + LoRA fits in
#         16 GB.  Eliminates every cross-device indexing RuntimeError.
# ── Device ─────────────────────────────────────────────────────────────────
#DEVICE      = "cuda:0"          # input tensors always land on first device
DEVICE_MAP  = "balanced_low_0"  # FIX E: keeps cuda:0 lighter for activation spikes
MAX_MEMORY  = {0: "13GiB", 1: "14GiB", "cpu": "24GiB"}  # FIX E: headroom per GPU


# ── Training hyperparams ───────────────────────────────────────────────────
EPOCHS             = 1
MICRO_BATCH        = 1          # per-step forward batch
GRAD_ACCUM         = 16         # effective batch = MICRO_BATCH × GRAD_ACCUM
LR_LLM             = 2e-4       # LoRA LR for the LLM backbone
LR_VIT             = 2e-5       # LoRA LR for the ViT encoder (gentle nudge)
LR_CONNECTOR       = 4e-5       # mlp1 connector full fine-tune LR
WEIGHT_DECAY       = 0.01
MAX_GRAD_NORM      = 1.0
WARMUP_RATIO       = 0.05
MAX_SEQ_LEN        = 2048       # token budget per sample (fits in 16 GB)
SAVE_EVERY_N_STEPS = 50
VAL_SPLIT          = 0.1
RANDOM_SEED        = 42

# ── QLoRA ──────────────────────────────────────────────────────────────────
LORA_R             = 16
LORA_ALPHA         = 32
LORA_DROPOUT       = 0.05

# With this:
LLM_TARGET_MODULES = ["wqkv", "wo", "w1", "w2", "w3"]
VIT_TARGET_MODULES = []             # FIX C: no ViT LoRA — saves ~3 GB activation retention

# ── Frame sampling ─────────────────────────────────────────────────────────
NUM_FRAMES      = 4            # FIX B: 4×256=1024 img tokens; fits well under MAX_SEQ_LEN=2048
FRAME_INPUT_SIZE = 224          # spatial resolution fed to ViT

# FIX 4: NUM_IMAGE_TOKEN is set at runtime from model.num_image_token.
#         DO NOT hardcode — it must match the mlp1 connector output exactly.
NUM_IMAGE_TOKEN  = None          # populated in load_model_with_qlora()
_PAD_TOKEN_ID   = 0               # FIX F: set from tokenizer in load_model_with_qlora()

# ── Question keys ──────────────────────────────────────────────────────────
CONTEXT_Q_KEYS = [
    "Q1a", "Q1b", "Q2", "Q3a", "Q3b",
    "Q4a", "Q4b", "Q4c", "Q9a", "Q9b",
]
TARGET_Q_KEYS = [
    "Q5a", "Q5b", "Q5c", "Q5e",
    "Q5f", "Q5g", "Q5i", "Q5j",
]

# ── System prompt ──────────────────────────────────────────────────────────
SYSTEM_PROMPT = (
    "You are an expert traffic accident analyst reviewing dashcam footage. "
    "The ego-car is the vehicle whose dashcam recorded the video."
)


# ╔═════════════════════════════════════════════════════════════════════════╗
# ║                    ❷  LABEL DEFINITIONS                                 ║
# ╚═════════════════════════════════════════════════════════════════════════╝

ALL_INCIDENT_QUESTIONS = [
    ("Q1a", "Time of day",
     {0: "Day", 1: "Dawn (sunrise)", 2: "Dusk (sunset)", 3: "Night", -1: "Unknown"}),
    ("Q1b", "Weather conditions",
     {0: "Sunny/Clear", 1: "Cloudy", 2: "Rainy", 3: "Partly Cloudy",
      4: "Snowy", 5: "Foggy", -1: "Unknown"}),
    ("Q2", "Lighting condition",
     {0: "Daylight", 1: "Headlights only",
      2: "Sunrise/Sunset – Early morning low-light, sun beginning to rise",
      4: "Streetlights on", -1: "Unknown"}),
    ("Q3a", "Traffic environment",
     {0: "Urban area – City streets with dense buildings, intersections, and traffic",
      2: "Suburban area – Residential or mixed-use commercial/residential area",
      3: "Rural area – Open or undeveloped roadside with sparse traffic",
      -1: "Unknown"}),
    ("Q3b", "Surrounding facilities",
     {1: "Urban area - skyscrapers, businesses, etc.",
      2: "Construction zone / Work area – Road partially blocked or altered due to work",
      3: "Residential Area", 4: "Parking lot / Gas station",
      5: "Near a school/college – Educational zone with potential pedestrian activity",
      -1: "Unknown"}),
    ("Q4a", "Road configuration",
     {1: "Highway", 2: "Intersection", 3: "T-junction / Y-junction",
      4: "Residential street", 5: "Bridge / Overpass / Underpass",
      6: "Roundabout / Traffic circle", 7: "Exit/Entrance ramp", -1: "Unknown"}),
    ("Q4b", "Lane configuration",
     {0: "Two-way divided", 1: "Two-way not divided", 2: "One-way", -1: "Unknown"}),
    ("Q4c", "Ego-car lane direction",
     {0: "Right", 1: "Left", 2: "Middle", -1: "Unknown"}),
    ("Q5a", "Incident category",
     {0: "Pedestrian/Cyclist/Vulnerable Road Users",
      1: "Vehicle-to-Vehicle", 2: "Animals / Objects", 3: "No Collision",
      5: "Barriers / Fixed Objects", 6: "Vehicle Loss of Control / Rollover",
      -1: "Unknown"}),
    ("Q5b", "Primary Incident Entity",
     {0: "Pedestrian", 1: "Ego-car", 2: "Animal", 3: "No collision",
      5: "Another Vehicle", 6: "Cyclist", 10: "Flying Object",
      13: "Smoke", 29: "Object on the road", -1: "Unknown"}),
    ("Q5c", "Primary entity vehicle type",
     {0: "Small Car", 1: "No collision", 2: "Truck", 3: "Bicycle",
      4: "Bicycle Cart", 5: "Motorcycle", 6: "Scooter",
      8: "SUV", 9: "Bus", -2: "Not applicable", -1: "Unknown"}),
    ("Q5e", "Primary entity behavior",
     {0: "Crossing", 1: "Speeding", 3: "Driving normally",
      4: "Overtaking/Changing Lanes", 6: "Turning", 7: "Walking",
      8: "Stopped", 10: "Swerving", 11: "Ignoring traffic signal",
      12: "Lost control", 13: "Fell Down", 14: "Rolling through",
      15: "Failure to yield", 16: "Flying", 17: "Rolling over",
      18: "Stationary", 19: "Fallen into the road", -1: "Unknown"}),
    ("Q5f", "Secondary Incident Entity",
     {0: "Cyclist", 1: "Ego-car", 2: "Another Vehicle",
      3: "Object or Barrier", 4: "Pedestrian", 5: "Scooter",
      6: "Animal", -1: "Unknown"}),
    ("Q5g", "Secondary entity vehicle type",
     {0: "Scooter", 1: "Small Car", 2: "Bicycle", 3: "Truck",
      4: "SUV", 5: "Bus", -2: "Not applicable", -1: "Unknown"}),
    ("Q5i", "Secondary entity behavior",
     {0: "Driving normally", 1: "Swerving", 2: "Fixed Position",
      3: "Speeding", 4: "Crossing", 5: "Turning", 6: "Lost control",
      7: "Failure to yield", 8: "Flying", 9: "Stopped",
      10: "Fallen onto road", 11: "Overtaking/Changing Lanes",
      13: "Rolling over", 14: "UTurn", -1: "Unknown"}),
    ("Q5j", "Incident peak",
     {0: "Near-miss", 1: "No collision", 2: "Collision",
      3: "No effect", 4: "Avoided in advance",
      5: "Chain-reaction", -1: "Unknown"}),
    ("Q9a", "Road surface condition",
     {0: "Dry pavement", 1: "Wet road", 2: "Snow / Ice covered",
      3: "Debris / Mud on road", -1: "Unknown"}),
    ("Q9b", "Road material",
     {0: "Asphalt / Blacktop", 1: "Concrete",
      3: "Gravel / Dirt", -1: "Unknown"}),
]

Q_DEFS   = {k: (desc, codes) for k, desc, codes in ALL_INCIDENT_QUESTIONS}
CODE2LBL = {k: codes       for k, _,    codes in ALL_INCIDENT_QUESTIONS}

CSV_COL_TO_KEY = {
    "A)Weather:\nQ1.a: What was the time of day during the incident?":                    "Q1a",
    "A)Weather:\nQ1.b: What were the weather conditions during the incident?":            "Q1b",
    "B) Light Condition:\nQ2: What was the lighting condition at the time of the incident?": "Q2",
    "C) Traffic Environment:\nQ3.a: Where did the incident take place (traffic environment)?": "Q3a",
    "C) Traffic Environment:\nQ3.b: What facilities were present in the surrounding environment?": "Q3b",
    "D) Road Configuration\nQ4.a: What is the road configuration at the incident time?":  "Q4a",
    "D) Road Configuration\nQ4.b: Road lane configuration:":                              "Q4b",
    "D) Road Configuration\nQ4.c: Lane direction of Ego-Car at time of incident:":        "Q4c",
    "E) Incident Type\nQ5.a: What is the category of the incident?":                      "Q5a",
    "E) Incident Type\nQ5.b: Primary Incident Entity (Who caused or could have prevented the incident)?": "Q5b",
    "E) Incident Type\nQ5.c: If Primary Incident Entity is a vehicle, what type of vehicle is it?": "Q5c",
    "E) Incident Type\nQ5.e: Primary Entity Behavior:":                                   "Q5e",
    "E) Incident Type\nQ5.f Secondary Incident Entity (Victim, secondary cause or entity that was hit):": "Q5f",
    "E) Incident Type:\nQ5.g: If Secondary Incident Entity is a vehicle, what type of vehicle is it?": "Q5g",
    "E) Incident Type\nQ5.i: Secondary Entity Behavior:":                                 "Q5i",
    "E) Incident Type\nQ5.j: Incident peak":                                              "Q5j",
    "I) Road Surface & Material Condition\nQ9.a: What was the road surface condition during the incident?": "Q9a",
    "I) Road Surface & Material Condition\nQ9.b: What type of road material is the vehicle driving on?": "Q9b",
}


# ╔═════════════════════════════════════════════════════════════════════════╗
# ║                    ❸  DATA PREPARATION                                  ║
# ╚═════════════════════════════════════════════════════════════════════════╝

def load_and_decode_answers(answer_csv: str, reference_csv: str) -> pd.DataFrame:
    """Load answer.csv, map columns → Q-keys, decode numeric codes → text labels."""
    df_ans = pd.read_csv(answer_csv)
    df_ref = pd.read_csv(reference_csv)

    df_ans.columns = [c.strip() for c in df_ans.columns]
    vid_col = df_ans.columns[0]
    df_ans  = df_ans.rename(columns={vid_col: "video_id"})
    df_ans["video_id"] = df_ans["video_id"].astype(int)

    ref_ids = set(df_ref.iloc[:, 0].astype(int).tolist())
    df_ans  = df_ans[df_ans["video_id"].isin(ref_ids)].reset_index(drop=True)

    rename_map = {col: key for col, key in CSV_COL_TO_KEY.items() if col in df_ans.columns}
    df_ans = df_ans.rename(columns=rename_map)

    all_q_keys = CONTEXT_Q_KEYS + TARGET_Q_KEYS
    for q_key in all_q_keys:
        if q_key not in df_ans.columns:
            print(f"  WARNING: {q_key} not found — defaulting to 'Unknown'")
            df_ans[q_key] = -1

    for q_key in all_q_keys:
        code_map = CODE2LBL.get(q_key, {})
        def _decode(val, cm=code_map):
            try:
                return cm.get(int(float(val)), "Unknown")
            except Exception:
                return "Unknown"
        df_ans[q_key] = df_ans[q_key].apply(_decode)

    print(f"  Loaded {len(df_ans)} labelled samples.")
    return df_ans[["video_id"] + all_q_keys]


def build_context_string(row: pd.Series) -> str:
    """Build scene-condition context from A/B/C/D/I answers."""
    lines = ["Known scene conditions (use as factual context):"]
    for q_key in CONTEXT_Q_KEYS:
        desc  = Q_DEFS[q_key][0]
        label = row.get(q_key, "Unknown")
        lines.append(f"  - {desc}: {label}")
    return "\n".join(lines)


def build_target_json(row: pd.Series) -> str:
    """Build the ground-truth JSON that the model must learn to produce."""
    return json.dumps({q: row.get(q, "Unknown") for q in TARGET_Q_KEYS}, indent=2)


def build_user_prompt(context_str: str, num_frames: int) -> str:
    """
    Build the user-turn prompt.
    Each <image> placeholder will be replaced by NUM_IMAGE_TOKEN × <IMG_CONTEXT>
    tokens during tokenisation.  num_frames must equal len(pixel_values).
    """
    frame_lines = "\n".join(f"Frame{i+1}: <image>" for i in range(num_frames))

    q_lines = []
    for q_key in TARGET_Q_KEYS:
        opts = " | ".join(Q_DEFS[q_key][1].values())
        q_lines.append(f"[{q_key}] Options: {opts}")

    return (
        f"{frame_lines}\n\n"
        f"{context_str}\n\n"
        "You are an expert dashcam incident analyst.\n"
        "The ego-car is the vehicle whose dashcam recorded this video.\n"
        "CRITICAL: Carefully distinguish whether the ego-car itself is directly\n"
        "involved in any collision/incident, or if it is only a witness.\n"
        "Signs EGO-CAR IS INVOLVED: impact on windshield/bumper, sudden\n"
        "deceleration just before contact, entity at bumper level.\n"
        "Signs EGO-CAR IS A WITNESS: incident occurs far ahead between others.\n\n"
        "Using the video frames AND the known scene conditions above, answer ONLY\n"
        "the following incident-type questions.\n"
        "Respond with ONLY a valid JSON object — no markdown, no explanation.\n\n"
        + "\n".join(q_lines)
        + "\n\nAnswer format:\n{\n"
        + ",\n".join(f'  "{k}": "<exact option text>"' for k in TARGET_Q_KEYS)
        + "\n}"
    )


# ╔═════════════════════════════════════════════════════════════════════════╗
# ║                    ❹  FRAME LOADING                                     ║
# ╚═════════════════════════════════════════════════════════════════════════╝

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)


def make_transform(input_size: int = FRAME_INPUT_SIZE) -> T.Compose:
    return T.Compose([
        T.Lambda(lambda img: img.convert("RGB") if img.mode != "RGB" else img),
        T.Resize((input_size, input_size), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])


def load_frames(frames_dir: str, n_frames: int = NUM_FRAMES):
    """
    Uniformly sample up to n_frames JPEGs from frames_dir.
    Returns pixel_values  : FloatTensor [n, 3, H, W]
    """
    jpegs = sorted(glob.glob(os.path.join(frames_dir, "*.jpg")))
    if not jpegs:
        jpegs = sorted(glob.glob(os.path.join(frames_dir, "*.jpeg")))
    if not jpegs:
        raise FileNotFoundError(f"No JPEG frames in: {frames_dir}")

    if len(jpegs) > n_frames:
        indices = np.linspace(0, len(jpegs) - 1, n_frames, dtype=int)
        jpegs   = [jpegs[i] for i in indices]

    tfm = make_transform()
    return torch.stack([tfm(Image.open(p).convert("RGB")) for p in jpegs])


# ╔═════════════════════════════════════════════════════════════════════════╗
# ║                    ❺  TOKENISATION                                      ║
# ╚═════════════════════════════════════════════════════════════════════════╝

def build_training_tensors(
    tokenizer,
    user_prompt: str,
    assistant_response: str,
    num_frames: int,
    img_context_token_id: int,
    num_image_token: int,
):
    """
    Convert one sample into (input_ids, labels, image_flags).

    image_flags  — FIX 2: frame-level 1D tensor  [num_frames]  all 1s.
                   This matches the model forward:
                       vit_embeds = vit_embeds[image_flags == 1]
                   which expects one entry per frame, NOT one per token.

    The number of <IMG_CONTEXT> tokens inserted per frame = num_image_token.
    This is read from model.num_image_token at load time (FIX 4).
    """
    img_ctx_str = tokenizer.decode([img_context_token_id], skip_special_tokens=False)

    def _expand_image_placeholders(text: str) -> str:
        """Replace each <image> with num_image_token copies of <IMG_CONTEXT>."""
        return text.replace("<image>", img_ctx_str * num_image_token)

    prefix_raw   = (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{user_prompt}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    response_raw = f"{assistant_response}<|im_end|>\n"

    prefix_text   = _expand_image_placeholders(prefix_raw)
    response_text = response_raw  # no images in response

    prefix_ids   = tokenizer(prefix_text,   add_special_tokens=False).input_ids
    response_ids = tokenizer(response_text, add_special_tokens=False).input_ids
    full_ids     = (prefix_ids + response_ids)[:MAX_SEQ_LEN]
    prefix_len   = min(len(prefix_ids), MAX_SEQ_LEN)

    input_ids = torch.LongTensor(full_ids)

    # Loss only on assistant tokens
    labels = torch.full_like(input_ids, -100)
    labels[prefix_len:] = input_ids[prefix_len:]

    # FIX 2: frame-level flags — one 1 per frame, NOT per token
    image_flags = torch.ones(num_frames, dtype=torch.long)

    return input_ids, labels, image_flags


# ╔═════════════════════════════════════════════════════════════════════════╗
# ║                    ❻  DATASET                                           ║
# ╚═════════════════════════════════════════════════════════════════════════╝

class IncidentVQADataset(torch.utils.data.Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        tokenizer,
        img_context_token_id: int,
        num_image_token: int,
        frames_base_dir: str = SAMPLED_FRAMES_DIR,
    ):
        self.df                   = df.reset_index(drop=True)
        self.tokenizer            = tokenizer
        self.img_context_token_id = img_context_token_id
        self.num_image_token      = num_image_token      # from model config
        self.frames_base_dir      = frames_base_dir

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row    = self.df.iloc[idx]
        vid_id = int(row["video_id"])
        fdir   = os.path.join(self.frames_base_dir, str(vid_id))

        try:
            pixel_values = load_frames(fdir, NUM_FRAMES)  # [n, 3, H, W]
        except Exception as e:
            print(f"  [Dataset] Frame load failed for video {vid_id}: {e}")
            return None  # collate_fn will skip None

        n_frames = pixel_values.shape[0]

        context_str        = build_context_string(row)
        user_prompt        = build_user_prompt(context_str, n_frames)
        assistant_response = build_target_json(row)

        input_ids, labels, image_flags = build_training_tensors(
            tokenizer            = self.tokenizer,
            user_prompt          = user_prompt,
            assistant_response   = assistant_response,
            num_frames           = n_frames,
            img_context_token_id = self.img_context_token_id,
            num_image_token      = self.num_image_token,
        )

        return {
            "input_ids":    input_ids,       # [seq_len]
            "labels":       labels,          # [seq_len]
            "pixel_values": pixel_values,    # [n_frames, 3, H, W]
            "image_flags":  image_flags,     # [n_frames]  — frame-level
            "video_id":     vid_id,
        }


# ╔═════════════════════════════════════════════════════════════════════════╗
# ║                    ❼  COLLATE FUNCTION                                  ║
# ╚═════════════════════════════════════════════════════════════════════════╝

def collate_fn(batch):
    """
    Pad input_ids/labels to max seq-len in batch.
    FIX 3: image_flags is frame-level → torch.cat, NOT torch.stack.
    pixel_values: torch.cat along dim=0 → [total_frames, 3, H, W].
    """
    batch = [b for b in batch if b is not None]
    if not batch:
        return None

    max_len = max(b["input_ids"].shape[0] for b in batch)

    input_ids_list    = []
    labels_list       = []
    attention_mask_list = []  # FIX F
    pixel_values_list = []
    image_flags_list  = []

    for b in batch:
        seq_len = b["input_ids"].shape[0]
        pad_len = max_len - seq_len

        input_ids_list.append(
            torch.cat([b["input_ids"],
                       torch.full((pad_len,), _PAD_TOKEN_ID, dtype=torch.long)])  # FIX F
        )
        labels_list.append(
            torch.cat([b["labels"],
                       torch.full((pad_len,), -100, dtype=torch.long)])
        )
        # FIX F: build attention_mask — 1 for real tokens, 0 for padding
        attention_mask_list.append(
            torch.cat([torch.ones(seq_len, dtype=torch.long),
                       torch.zeros(pad_len, dtype=torch.long)])
        )
        pixel_values_list.append(b["pixel_values"])          # [n_frames, 3, H, W]
        image_flags_list.append(b["image_flags"])            # [n_frames]

    return {
        "input_ids":      torch.stack(input_ids_list),
        "labels":         torch.stack(labels_list),
        "attention_mask": torch.stack(attention_mask_list),              # FIX F: [B, L]
        "pixel_values":   torch.cat(pixel_values_list, dim=0),
        "image_flags":    torch.cat(image_flags_list,  dim=0),
    }

# ╔═════════════════════════════════════════════════════════════════════════╗
# ║                    ❼  HELPER FUNCTIONS                                  ║
# ╚═════════════════════════════════════════════════════════════════════════╝

# ── PATCH: replaced single get_input_device() with two focused helpers ───
# OLD get_input_device() always fell back to cuda:0 for everything.
# With DEVICE_MAP="balanced_low_0", ViT lands on cuda:0 but LLM embedding
# layer can land on cuda:1 depending on shard boundary.
# Routing all tensors to cuda:0 causes cross-device indexing RuntimeError.
# Fix: separate helpers for ViT device and LLM input device.

def get_vit_device(model):
    """Device of the ViT encoder — pixel_values MUST land here."""
    try:
        return next(model.vision_model.parameters()).device
    except StopIteration:
        return torch.device("cuda:0")


def get_llm_input_device(model):
    """Device of LLM embedding layer — input_ids / labels / image_flags go here."""
    try:
        emb = (getattr(model.language_model.model, "tok_embeddings", None)
               or getattr(model.language_model.model, "embed_tokens",  None))
        if emb is not None:
            return next(emb.parameters()).device
    except Exception:
        pass
    return torch.device("cuda:0")


# ── PATCH: source-level fix for dual-GPU cross-device embedding write ────
# The crash happens inside InternVLChatModel.forward():
#     input_embeds[selected] = vit_embeds
# With balanced_low_0, input_embeds migrates to cuda:1 as it passes through
# LLM layers, but `selected` (built from input_ids) stays on cuda:0.
# Writing a cuda:0-indexed slice into a cuda:1 tensor → RuntimeError.
# Fix: patch the cached model .py to add .to(input_embeds.device) so
# vit_embeds chases input_embeds to whichever GPU it currently lives on.
# Must be called BEFORE AutoModel.from_pretrained — trust_remote_code=True
# re-reads the cached .py file each time so the fix is picked up automatically.

def patch_internvl_source():
    cache_root = os.path.expanduser("~/.cache/huggingface/hub")
    pattern    = os.path.join(cache_root, "**", "modeling_internvl_chat*.py")
    files      = glob.glob(pattern, recursive=True)

    if not files:
        print("  WARNING: model source file not found — patch skipped.")
        print(f"  Searched: {cache_root}")
        return False

    TARGET  = "input_embeds[selected] = vit_embeds"
    PATCHED = "input_embeds[selected] = vit_embeds.to(input_embeds.device)"

    success = False
    for fpath in files:
        with open(fpath, "r") as f:
            src = f.read()

        if PATCHED in src:
            print(f"  ✓ Already patched: {os.path.basename(fpath)}")
            success = True
            continue

        if TARGET not in src:
            print(f"  WARNING: target line not found in {os.path.basename(fpath)}")
            print("  The model version may have changed — search manually for:")
            print(f"    {TARGET}")
            continue

        src = src.replace(TARGET, PATCHED)
        with open(fpath, "w") as f:
            f.write(src)
        print(f"  ✓ Patched: {fpath}")
        success = True

    return success
# ╔═════════════════════════════════════════════════════════════════════════╗
# ║                    ❽  MODEL LOADING + QLORA                             ║
# ╚═════════════════════════════════════════════════════════════════════════╝

def _find_existing_target_modules(model, wanted: list) -> list:
    """Return the subset of wanted module-name suffixes actually present in model."""
    found = set()
    for name, module in model.named_modules():
        for target in wanted:
            if name.endswith(target) and (
                isinstance(module, nn.Linear) or "Linear" in type(module).__name__
            ):
                found.add(target)
    return list(found)



def _enable_manual_gradient_checkpointing(model) -> int:
    """
    FIX A (revised) — manual per-layer gradient checkpointing.

    InternVLChatModel sets supports_gradient_checkpointing = False so
    model.gradient_checkpointing_enable() always raises:
        "InternVLChatModel does not support gradient checkpointing."
    Work-around: wrap each InternLM2 decoder-layer's forward() with
    torch.utils.checkpoint.checkpoint — identical memory savings, zero
    API conflict. use_reentrant=False avoids the in-place mutation warning
    and is required for compatibility with modern PyTorch autograd.
    """
    from torch.utils.checkpoint import checkpoint as ckpt_fn

    try:
        layers = model.language_model.model.layers
    except AttributeError:
        print("  ⚠ LLM decoder layers not found — manual grad ckpt skipped")
        return 0

    for layer in layers:
        orig = layer.forward
        def _make(orig=orig):
            def _ckpt_forward(*args, **kwargs):
                return ckpt_fn(orig, *args, use_reentrant=False, **kwargs)
            return _ckpt_forward
        layer.forward = _make()

    return len(layers)

def load_model_with_qlora():
    """
    Load InternVideo2.5 in 4-bit NF4, inject LoRA in-place, fix device placement.

    ALL FIXES:
    ──────────────────────────────────────────────────────────────────────────
    FIX 1 │ DEVICE_MAP="balanced_low_0" — dual GPU, ViT on cuda:0, LLM split
    FIX 2 │ image_flags shape [num_frames] not [seq_len] — in build_training_tensors
    FIX 3 │ torch.cat not torch.stack for image_flags — in collate_fn
    FIX 4 │ NUM_IMAGE_TOKEN read from model at runtime, never hardcoded
    FIX 5 │ LoRA params moved to same device as their base layer (not hardcoded cuda:0)
    FIX 6 │ prepare_model_for_kbit_training(use_gradient_checkpointing=False) — model forbids it
    FIX A │ _enable_manual_gradient_checkpointing() wraps each LLM decoder layer forward
    FIX 7 │ Nuclear dtype cast: all float32 trainable params → float16
    FIX 8 │ num_workers=0 in DataLoaders — in train()
    FIX 9 │ dtype guard in unfreeze loop — uint8 Params4bit cannot have requires_grad
    PATCH  │ patch_internvl_source() called before from_pretrained in train()
    PATCH  │ get_vit_device / get_llm_input_device replace hardcoded DEVICE routing
    ──────────────────────────────────────────────────────────────────────────
    """
    global NUM_IMAGE_TOKEN

    print(f"\n{'='*60}")
    print("  Loading model in 4-bit NF4 on dual GPU (balanced_low_0)…")
    print(f"{'='*60}")

    # ── 4-bit NF4 quantisation ──────────────────────────────────────────────
    # bnb_4bit_compute_dtype=float16 → all forward activations are float16.
    # Every floating-point trainable param must also be float16 (see FIX 7).
    # Quantized weights are stored as uint8 (Params4bit) — not float32.
    quant_cfg = BitsAndBytesConfig(
        load_in_4bit              = True,
        bnb_4bit_compute_dtype    = torch.float16,
        bnb_4bit_use_double_quant = True,
        bnb_4bit_quant_type       = "nf4",
    )

    # ── FIX 1: dual GPU with balanced_low_0 ────────────────────────────────
    # "balanced_low_0" spreads layers across both GPUs while keeping cuda:0
    # lighter (leaves room for activations + LoRA gradients on cuda:0).
    # ViT encoder always lands on cuda:0 (first layers loaded first).
    # LLM layers are split — embedding may land on cuda:0 or cuda:1.
    # get_vit_device() / get_llm_input_device() handle routing correctly.
    model = AutoModel.from_pretrained(
        MODEL_PATH,
        trust_remote_code   = True,
        device_map          = DEVICE_MAP,       # FIX E: "balanced_low_0"
        **({"max_memory": MAX_MEMORY} if MAX_MEMORY else {}),  # FIX E
        quantization_config = quant_cfg,
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
    global _PAD_TOKEN_ID
    _PAD_TOKEN_ID = tokenizer.pad_token_id or tokenizer.eos_token_id or 0  # FIX F

    # ── FIX 4: read NUM_IMAGE_TOKEN at runtime ──────────────────────────────
    # Must match mlp1 output token count exactly.
    # Hardcoding any value risks silent shape misalignment → crash in LLM.
    if hasattr(model, "num_image_token"):
        NUM_IMAGE_TOKEN = model.num_image_token
    else:
        cfg             = AutoConfig.from_pretrained(MODEL_PATH, trust_remote_code=True)
        img_size        = getattr(cfg, "force_image_size", None) \
                          or cfg.vision_config.image_size
        patch_size      = cfg.vision_config.patch_size
        down_ratio      = getattr(cfg, "downsample_ratio", 0.5)
        NUM_IMAGE_TOKEN = int((img_size // patch_size) ** 2 * (down_ratio ** 2))
    print(f"  NUM_IMAGE_TOKEN  : {NUM_IMAGE_TOKEN}  (tokens per frame)")

    img_ctx_id = tokenizer.convert_tokens_to_ids("<IMG_CONTEXT>")
    print(f"  IMG_CONTEXT id   : {img_ctx_id}")

    # ── FIX A (was FIX 6): gradient checkpointing ENABLED ────────────────────
    # Safe to enable now because init_single_gpu_process_group() (FIX 10) runs
    # before load_model_with_qlora(), giving InternLM2's dist.get_rank() a valid
    # gloo process group (rank=0, world_size=1). Saves ~4-6 GB of LLM activations.
    model = prepare_model_for_kbit_training(
        model,
        use_gradient_checkpointing=False,   # must stay False — InternVLChatModel forbids it
    )
    print("  kbit prep done (InternVLChatModel.supports_gradient_checkpointing=False)")

    model.img_context_token_id = img_ctx_id

    # ── Scan for actually-present LoRA target modules ───────────────────────
    vit_found   = _find_existing_target_modules(model.vision_model,   VIT_TARGET_MODULES)
    llm_found   = _find_existing_target_modules(model.language_model, LLM_TARGET_MODULES)
    all_targets = list(set(vit_found + llm_found))
    print(f"  ViT LoRA targets : {vit_found}")
    print(f"  LLM LoRA targets : {llm_found}")

    lora_cfg = LoraConfig(
        r              = LORA_R,
        lora_alpha     = LORA_ALPHA,
        lora_dropout   = LORA_DROPOUT,
        bias           = "none",
        target_modules = all_targets,
        # No task_type — avoids PeftModel wrapper that overrides InternVL's
        # custom forward(pixel_values, image_flags, ...) signature.
    )

    # Inject LoRA directly into Linear layers (no PeftModel wrapper)
    model = inject_adapter_in_model(lora_cfg, model)

    # ── FIX A (revised): manual gradient checkpointing on LLM decoder layers ─
    # InternVLChatModel.supports_gradient_checkpointing = False means
    # model.gradient_checkpointing_enable() always raises. Work-around:
    # wrap each InternLM2 decoder-layer forward with torch.utils.checkpoint.
    # This gives the same activation-recompute savings without touching the
    # model's own checkpointing flag. Saves ~4-6 GB of LLM activation memory.
    _n_ckpt = _enable_manual_gradient_checkpointing(model)
    print(f"  ✓ Manual gradient checkpointing: {_n_ckpt} LLM decoder layers wrapped")

    # ── FIX 5 revised: per-layer LoRA device placement ─────────────────────
    # inject_adapter_in_model creates lora_A/lora_B on CPU for every layer.
    # With balanced_low_0, base layers are distributed across cuda:0 and cuda:1.
    # OLD FIX moved everything to cuda:0 — wrong for layers that live on cuda:1.
    # CORRECT FIX: for each module, find the device of its base weight and
    # move that module's LoRA params to the same device.
    print("  Placing LoRA params on correct device per layer…")
    n_cast, n_moved = 0, 0

    for module_name, module in model.named_modules():
        # Find the device of this module's base weight (first non-LoRA float param)
        base_device = None
        for pname, p in module.named_parameters(recurse=False):
            if "lora_" not in pname and p.device.type != "cpu":
                base_device = p.device
                break

        # Move LoRA params in this module to match its base weight device
        if base_device is not None:
            for pname, p in module.named_parameters(recurse=False):
                if "lora_" in pname and p.device.type == "cpu":
                    p.data = p.data.to(base_device)
                    n_moved += 1

    # ── FIX 7: cast any float32 trainable params to float16 ────────────────
    # prepare_model_for_kbit_training upcasts mlp1 biases / layernorm to float32.
    # 4-bit compute dtype is float16 → activations are float16.
    # A float32 bias causes:
    #   RuntimeError: Input type (c10::Half) and bias type (float) should be the same
    # Every trainable param must be float16 to match compute dtype.
    for name, param in model.named_parameters():
        if param.dtype == torch.float32:
            param.data = param.data.to(torch.float16)
            n_cast += 1

    # Catch any remaining CPU stragglers (e.g. mlp1 not in LoRA modules)
    for name, param in model.named_parameters():
        if param.device.type == "cpu":
            # mlp1 lives at model input boundary — cuda:0 is always safe here
            param.data = param.data.to("cuda:0")
            n_moved += 1

    print(f"  Params cast float32→float16  : {n_cast}")
    print(f"  Params moved CPU→correct GPU : {n_moved}")

    # ── Freeze all params ───────────────────────────────────────────────────
    # 4-bit quantized weights (uint8 Params4bit) MUST stay frozen —
    # bitsandbytes cannot compute gradients through quantized weights.
    for param in model.parameters():
        param.requires_grad = False

    # ── FIX 9: unfreeze LoRA + mlp1 with dtype guard ───────────────────────
    # mlp1 weights → uint8 Params4bit → cannot have requires_grad=True.
    # mlp1 biases  → float16 (not quantized) → safely trainable.
    # lora_A/B     → float16 (new params, not quantized) → safely trainable.
    # Without dtype guard PyTorch raises:
    #   RuntimeError: only Tensors of floating point dtype can require gradients
    for name, param in model.named_parameters():
        if "lora_" in name or "mlp1" in name:
            # FIX 9: skip uint8 quantized weights — only enable float params
            if param.dtype in (torch.float16, torch.float32, torch.bfloat16):
                param.requires_grad = True

    # ── Parameter summary ───────────────────────────────────────────────────
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    vit_lora  = sum(p.numel() for n, p in model.named_parameters()
                    if "vision_model"   in n and "lora_" in n and p.requires_grad)
    llm_lora  = sum(p.numel() for n, p in model.named_parameters()
                    if "language_model" in n and "lora_" in n and p.requires_grad)
    conn      = sum(p.numel() for n, p in model.named_parameters()
                    if "mlp1" in n and p.requires_grad)

    print(f"\n  Trainable params  : {trainable:,}  ({100*trainable/total:.3f}%)")
    print(f"  ViT  LoRA params  : {vit_lora:,}")
    print(f"  LLM  LoRA params  : {llm_lora:,}")
    print(f"  Connector params  : {conn:,}")

    # ── Sanity checks: dtype and device ─────────────────────────────────────
    bad_dtype = [
        (n, p.dtype) for n, p in model.named_parameters()
        if p.requires_grad and p.dtype not in (torch.float16, torch.bfloat16)
    ]
    # PATCH: any cuda device is fine — only flag CPU stragglers
    bad_device = [
        (n, p.device) for n, p in model.named_parameters()
        if p.requires_grad and p.device.type == "cpu"
    ]

    if bad_dtype:
        print(f"\n  ⚠ WARNING — {len(bad_dtype)} trainable params wrong dtype:")
        for n, d in bad_dtype[:5]:
            print(f"    {n}: {d}")
        print("    → Will cause 'Input type c10::Half and bias type float' error")
    else:
        print("  ✓ dtype check passed — all trainable params float16  (FIX 7+9)")

    if bad_device:
        print(f"\n  ⚠ WARNING — {len(bad_device)} trainable params on CPU:")
        for n, d in bad_device[:5]:
            print(f"    {n}: {d}")
        print("    → Will cause 'Expected all tensors on same device' error")
    else:
        print("  ✓ device check passed — all trainable params on GPU  (FIX 5 revised)")

    return model, tokenizer, img_ctx_id


# ╔═════════════════════════════════════════════════════════════════════════╗
# ║                    ❾  CHECKPOINT SAVE / LOAD                            ║
# ╚═════════════════════════════════════════════════════════════════════════╝

def save_lora_checkpoint(model, tokenizer, path: str):
    """Save LoRA + connector weights only (compact, no base model)."""
    os.makedirs(path, exist_ok=True)
    lora_state = {
        name: param.cpu()
        for name, param in model.named_parameters()
        if "lora_" in name or "mlp1" in name
    }
    torch.save(lora_state, os.path.join(path, "lora_weights.pt"))
    tokenizer.save_pretrained(path)
    with open(os.path.join(path, "lora_config.json"), "w") as f:
        json.dump({
            "r":              LORA_R,
            "lora_alpha":     LORA_ALPHA,
            "lora_dropout":   LORA_DROPOUT,
            "num_image_token": NUM_IMAGE_TOKEN,
        }, f, indent=2)
    print(f"  Saved → {path}  ({len(lora_state)} tensors)")


def load_finetuned_model(checkpoint_path: str):
    """Re-load base model, re-inject LoRA structure, restore saved weights."""
    model, tokenizer, img_ctx_id = load_model_with_qlora()

    weights_path = os.path.join(checkpoint_path, "lora_weights.pt")
    lora_state   = torch.load(weights_path, map_location="cpu")

    # Move loaded weights to DEVICE before loading into model
    lora_state = {k: v.to("cuda:0") for k, v in lora_state.items()}  # FIX J: DEVICE was undefined
    missing, unexpected = model.load_state_dict(lora_state, strict=False)
    print(f"  Loaded weights — missing: {len(missing)}  unexpected: {len(unexpected)}")

    model.eval()
    return model, tokenizer


# ╔═════════════════════════════════════════════════════════════════════════╗
# ║                    ❿  OPTIMIZER                                         ║
# ╚═════════════════════════════════════════════════════════════════════════╝

def build_optimizer(model):
    vit_params, conn_params, llm_params = [], [], []

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if "vision_model" in name:
            vit_params.append(param)
        elif "mlp1" in name:
            conn_params.append(param)
        else:
            llm_params.append(param)

    print(f"\n  Optimiser groups:")
    print(f"    ViT  LoRA  : {sum(p.numel() for p in vit_params):>10,}  lr={LR_VIT}")
    print(f"    Connector  : {sum(p.numel() for p in conn_params):>10,}  lr={LR_CONNECTOR}")
    print(f"    LLM  LoRA  : {sum(p.numel() for p in llm_params):>10,}  lr={LR_LLM}")

    # FIX D: PagedAdamW8bit — 4× less optimizer-state VRAM + paged CPU spill safety net
    return bnb.optim.PagedAdamW8bit(
        [
            {"params": vit_params,  "lr": LR_VIT,       "weight_decay": WEIGHT_DECAY},
            {"params": conn_params, "lr": LR_CONNECTOR, "weight_decay": WEIGHT_DECAY},
            {"params": llm_params,  "lr": LR_LLM,       "weight_decay": WEIGHT_DECAY},
        ],
        betas=(0.9, 0.999),
        eps=1e-8,
    )


def build_scheduler(optimizer, total_steps: int):
    warmup_steps = max(1, int(total_steps * WARMUP_RATIO))

    def lr_lambda(step):
        if step < warmup_steps:
            return float(step) / float(max(1, warmup_steps))
        progress = float(step - warmup_steps) / float(max(1, total_steps - warmup_steps))
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda), warmup_steps


# ╔═════════════════════════════════════════════════════════════════════════╗
# ║                    ⓫  TRAINING LOOP                                     ║
# ╚═════════════════════════════════════════════════════════════════════════╝

def run_validation(model, val_loader):
    model.eval()
    total_loss, n = 0.0, 0

    # PATCH: compute device targets once outside the loop (cheaper than per-batch).
    # get_vit_device / get_llm_input_device replace the old hardcoded .to(DEVICE).
    # With balanced_low_0, ViT is always cuda:0 but LLM input may be cuda:1.
    vit_dev = get_vit_device(model)
    llm_dev = get_llm_input_device(model)

    with torch.no_grad():
        for batch in tqdm(val_loader, desc="  Validation", leave=False):
            if batch is None:
                continue
            try:
                outputs = model(
                    input_ids      = batch["input_ids"].to(llm_dev),
                    labels         = batch["labels"].to(llm_dev),
                    attention_mask = batch["attention_mask"].to(llm_dev),   # FIX F
                    pixel_values   = batch["pixel_values"].to(vit_dev, dtype=torch.float16, non_blocking=True),  # FIX G
                    image_flags    = batch["image_flags"].to(llm_dev),
                    use_cache      = False,
                )
                total_loss += outputs.loss.item()
                n += 1
            except Exception as e:
                print(f"  Val error: {e}")
    model.train()
    return total_loss / max(n, 1)


import torch.distributed as dist

def init_single_gpu_process_group():
    """FIX 10 —
    InternLM2 calls dist.get_rank() / get_world_size() unconditionally in its
    attention forward pass (XTuner sequence-parallel code baked into the source).
    Without an initialized process group every forward crashes:
      RuntimeError: Default process group has not been initialized
    Fix: initialize a trivial gloo group (rank=0, world=1).
    Zero communication overhead. gloo backend is CPU-based — no NCCL needed.
    """
    if not dist.is_initialized():
        os.environ.setdefault("MASTER_ADDR", "127.0.0.1")
        os.environ.setdefault("MASTER_PORT", "29500")
        dist.init_process_group(
            backend    = "gloo",
            rank       = 0,
            world_size = 1,
        )
        print("  ✓ Initialized single-process group (gloo, rank=0, world_size=1)")
    else:
        print("  ✓ Process group already initialized — skipping.")


def train():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    random.seed(RANDOM_SEED)
    torch.manual_seed(RANDOM_SEED)

    # ── FIX 10: Must be FIRST — before any model load or forward ─────────
    init_single_gpu_process_group()

    # ── PATCH: patch InternVL source BEFORE from_pretrained loads it ──────
    # trust_remote_code=True re-reads the cached .py file each run, so the
    # one-line fix (vit_embeds.to(input_embeds.device)) is picked up automatically.
    # If the patch fails (file not found / line already changed), fall back to
    # single-GPU mode to avoid the cross-device embedding crash entirely.
    print("\n[0/4] Patching InternVL source for dual-GPU compatibility…")
    patched = patch_internvl_source()
    if not patched:
        print("  ⚠ Patch failed — falling back to single-GPU (DEVICE_MAP={'': 0})")
        global DEVICE_MAP, MAX_MEMORY
        DEVICE_MAP = {"": 0}
        MAX_MEMORY = None

    # ── 1. Data ───────────────────────────────────────────────────────────
    print("\n[1/4] Loading data…")
    df      = load_and_decode_answers(ANSWER_CSV_PATH, REFERENCE_CSV_PATH)
    df      = df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
    n_val   = max(1, int(len(df) * VAL_SPLIT))
    df_val  = df.iloc[:n_val].reset_index(drop=True)
    df_trn  = df.iloc[n_val:].reset_index(drop=True)
    print(f"  Train: {len(df_trn)}  |  Val: {len(df_val)}")

    # ── 2. Model ──────────────────────────────────────────────────────────
    print("\n[2/4] Loading model + QLoRA…")
    model, tokenizer, img_ctx_id = load_model_with_qlora()

    # ── 3. Datasets & DataLoaders ─────────────────────────────────────────
    print("\n[3/4] Building datasets…")
    ds_kwargs = dict(
        tokenizer            = tokenizer,
        img_context_token_id = img_ctx_id,
        num_image_token      = NUM_IMAGE_TOKEN,   # FIX 4: pass runtime value
    )
    train_ds = IncidentVQADataset(df_trn, **ds_kwargs)
    val_ds   = IncidentVQADataset(df_val, **ds_kwargs)

    # ── FIX 8: num_workers=0 ───────────────────────────────────────────────
    # num_workers > 0 forks child processes. In Kaggle/Jupyter the parent PID
    # check in multiprocessing fails with AssertionError: can only test a child
    # process when the DataLoader tries to monitor worker health via w.is_alive().
    # num_workers=0 runs data loading in the main process — no forking, no crash.
    train_loader = torch.utils.data.DataLoader(
        train_ds,
        batch_size  = MICRO_BATCH,
        shuffle     = True,
        num_workers = 0,           # FIX 8
        collate_fn  = collate_fn,
        pin_memory  = True,   # FIX G: faster H2D with non_blocking copies
    )
    val_loader = torch.utils.data.DataLoader(
        val_ds,
        batch_size  = MICRO_BATCH,
        shuffle     = False,
        num_workers = 0,           # FIX 8
        collate_fn  = collate_fn,
        pin_memory  = True,   # FIX G
    )

    # ── 4. Optimiser & Scheduler ──────────────────────────────────────────
    optimizer    = build_optimizer(model)
    total_steps  = max(1, math.ceil(len(train_loader) / GRAD_ACCUM) * EPOCHS)
    scheduler, warmup_steps = build_scheduler(optimizer, total_steps)
    print(f"\n  Total optimiser steps : {total_steps}")
    print(f"  Warmup steps          : {warmup_steps}")

    # ── 5. Training loop ──────────────────────────────────────────────────
    print(f"\n[4/4] Training — {EPOCHS} epoch(s), {total_steps} steps total")
    print(f"  Grad accum : {GRAD_ACCUM}  |  Effective batch: {GRAD_ACCUM * MICRO_BATCH}\n")

    model.train()
    best_val_loss  = float("inf")
    global_step    = 0
    trainable_params = [p for p in model.parameters() if p.requires_grad]  # FIX I: cache once

    # PATCH: compute device targets once per training run — they never change
    # after model load. Avoids redundant parameter scans inside the hot loop.
    vit_dev = get_vit_device(model)
    llm_dev = get_llm_input_device(model)
    print(f"  ViT device       : {vit_dev}   (pixel_values routed here)")
    print(f"  LLM input device : {llm_dev}   (input_ids / labels / image_flags routed here)")

    for epoch in range(EPOCHS):
        epoch_loss, n_batches = 0.0, 0
        optimizer.zero_grad()

        for batch_idx, batch in enumerate(
            tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
        ):
            if batch is None:
                continue

            # ── PATCH: route tensors to correct GPU ───────────────────────
            # With balanced_low_0: ViT → cuda:0, LLM input → cuda:0 or cuda:1.
            # pixel_values MUST go to the ViT device.
            # input_ids / labels / image_flags MUST go to the LLM input device.
            # image_flags follows input_ids: both used in the same indexing
            # operation inside InternVLChatModel.forward().
            # The source-level patch (patch_internvl_source) fixes the line:
            #   input_embeds[selected] = vit_embeds
            # so vit_embeds chases input_embeds across devices automatically.
            input_ids      = batch["input_ids"].to(llm_dev)
            labels         = batch["labels"].to(llm_dev)
            attention_mask = batch["attention_mask"].to(llm_dev)  # FIX F
            pixel_values   = batch["pixel_values"].to(vit_dev, dtype=torch.float16, non_blocking=True)  # FIX G
            image_flags    = batch["image_flags"].to(llm_dev)

            try:
                outputs = model(
                    input_ids      = input_ids,
                    labels         = labels,
                    attention_mask = attention_mask,  # FIX F
                    pixel_values   = pixel_values,
                    image_flags    = image_flags,
                    use_cache      = False,
                )
                loss = outputs.loss / GRAD_ACCUM

                if torch.isnan(loss) or torch.isinf(loss):
                    print(f"  WARNING: NaN/Inf loss at step {global_step} — skipping.")
                    optimizer.zero_grad()
                    continue

                loss.backward()
                epoch_loss += loss.item() * GRAD_ACCUM
                n_batches  += 1

            except torch.cuda.OutOfMemoryError:
                print(f"  OOM at batch {batch_idx} — skipping, clearing cache.")
                optimizer.zero_grad()
                torch.cuda.empty_cache()
                gc.collect()
                continue
            except Exception as e:
                print(f"  Forward error at batch {batch_idx}: {e}")
                optimizer.zero_grad()
                continue
            finally:
                del input_ids, labels, attention_mask, pixel_values, image_flags

            # ── Gradient step ──────────────────────────────────────────────
            if (batch_idx + 1) % GRAD_ACCUM == 0:
                torch.nn.utils.clip_grad_norm_(trainable_params, MAX_GRAD_NORM)  # FIX I
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
                global_step += 1

                if global_step % 10 == 0:
                    lr_now = scheduler.get_last_lr()
                    avg    = epoch_loss / max(n_batches, 1)
                    print(
                        f"  Step {global_step:4d} | loss {avg:.4f} | "
                        f"lr_llm {lr_now[-1]:.2e} | "
                        f"mem0 {torch.cuda.memory_allocated(0)/1e9:.1f} GB | "
                        f"mem1 {torch.cuda.memory_allocated(1)/1e9:.1f} GB"
                        # PATCH: log both GPUs so you can spot imbalance early
                    )

                if global_step % SAVE_EVERY_N_STEPS == 0:
                    save_lora_checkpoint(
                        model, tokenizer,
                        os.path.join(OUTPUT_DIR, f"step_{global_step}"),
                    )

            # FIX H: per-batch empty_cache() removed — causes fragmentation OOM

        # ── End-of-epoch validation ─────────────────────────────────────────
        avg_val = run_validation(model, val_loader)
        avg_trn = epoch_loss / max(n_batches, 1)
        print(f"\n  Epoch {epoch+1} — train_loss: {avg_trn:.4f}  val_loss: {avg_val:.4f}")

        if avg_val < best_val_loss:
            best_val_loss = avg_val
            best_path     = os.path.join(OUTPUT_DIR, "best_checkpoint")
            save_lora_checkpoint(model, tokenizer, best_path)
            print(f"  ✓ New best val_loss={best_val_loss:.4f} → {best_path}")

        model.train()
        gc.collect()
        torch.cuda.empty_cache()

    # ── Final save ─────────────────────────────────────────────────────────
    final_path = os.path.join(OUTPUT_DIR, "final_checkpoint")
    save_lora_checkpoint(model, tokenizer, final_path)
    print(f"\n  Final checkpoint saved: {final_path}")
    print("  Training complete ✓")


# ╔═════════════════════════════════════════════════════════════════════════╗
# ║                    ⓬  ENTRY POINT                                       ║
# ╚═════════════════════════════════════════════════════════════════════════╝

if __name__ == "__main__":
    train()

In [ ]:
# # ── Entry point ──────────────────────────────────────────────────────────────
# if __name__ == "__main__":
#     train()
 